# 05 — Theft / Unauthorized Movement Detection

Identify periods of suspicious motion in data recorded while the scooter
should be parked (engine off). Any sustained `gyr_magnitude` above a
stationary threshold triggers a theft alert.

In [1]:
from pathlib import Path
import plotly.graph_objects as go

from scooter_imu.data.loader import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.theft_detection import TheftDetector

RAW_PATH = Path("../data/raw/ScooterIMUData.csv")

In [2]:
df = preprocess(load_raw(RAW_PATH))

# For demo: treat the entire dataset as a 'parked' period
result = TheftDetector(threshold=0.01, window_s=2.0).predict(df)
print(f"Theft alert rows: {result['theft_alert'].sum()}")
result[["timestamp", "gyr_magnitude", "is_moving", "movement_ratio", "theft_alert"]].head(20)

Theft alert rows: 3000


,timestamp,gyr_magnitude,is_moving,movement_ratio,theft_alert
0,2024-03-15 10:00:00.000,10.796508,True,1.0,True
1,2024-03-15 10:00:00.020,10.051419,True,1.0,True
2,2024-03-15 10:00:00.040,9.114419,True,1.0,True
3,2024-03-15 10:00:00.060,12.103252,True,1.0,True
4,2024-03-15 10:00:00.080,10.395432,True,1.0,True
5,2024-03-15 10:00:00.100,5.230111,True,1.0,True
6,2024-03-15 10:00:00.120,7.883137,True,1.0,True
7,2024-03-15 10:00:00.140,4.263000,True,1.0,True
8,2024-03-15 10:00:00.160,8.212884,True,1.0,True
9,2024-03-15 10:00:00.180,6.871330,True,1.0,True


In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=result["timestamp"], y=result["gyr_magnitude"],
                         name="Magnitude", line=dict(color="gray", width=1)))
fig.add_trace(go.Scatter(x=result["timestamp"], y=result["movement_ratio"],
                         name="Movement Ratio", line=dict(color="orange", width=1.5)))

alerts = result[result["theft_alert"]]
fig.add_trace(go.Scatter(x=alerts["timestamp"], y=alerts["gyr_magnitude"],
                         mode="markers", name="Theft Alert",
                         marker=dict(color="red", size=6)))
fig.add_hline(y=0.01, line_dash="dash", line_color="red", annotation_text="Threshold")
fig.update_layout(title="Theft Detection — Movement Signal", xaxis_title="Time")
fig.show()